## Code for Wise Case
[Job Position](https://wise.jobs/job/senior-product-analyst-regional-expansion-in-sao-paulo-jid-1656)

##### Bootstrap the data

In [236]:
import pandas as pd
import numpy as np
import plotly.graph_objects as go

file_path = "../data/wise_funnel_events_regional_v2.csv"
df = pd.read_csv(file_path)

### EDA

In [237]:
# Understand schema, missing values, date coverage, and the main categorical values
# before calculating metrics.

# Copy to avoid accidental mutation of raw data
raw_df = df.copy()

raw_df.info()
raw_df.describe(include='all')

<class 'pandas.DataFrame'>
RangeIndex: 73440 entries, 0 to 73439
Data columns (total 8 columns):
 #   Column           Non-Null Count  Dtype
---  ------           --------------  -----
 0   event_name       73440 non-null  str  
 1   dt               73440 non-null  str  
 2   source_currency  73440 non-null  str  
 3   target_currency  73440 non-null  str  
 4   user_id          73440 non-null  int64
 5   user_country     73440 non-null  str  
 6   platform         73440 non-null  str  
 7   experience       73440 non-null  str  
dtypes: int64(1), str(7)
memory usage: 4.5 MB


,event_name,dt,source_currency,target_currency,user_id,user_country,platform,experience
count,73440,73440,73440,73440,7.344000e+04,73440,73440,73440
unique,3,61,1,1,NaN,3,3,2
top,Transfer Created,2024-02-24,MXN,USD,NaN,USA,Android,New
freq,43070,1592,73440,73440,NaN,28307,27593,39812
mean,NaN,NaN,NaN,NaN,1.501395e+06,NaN,NaN,NaN
std,NaN,NaN,NaN,NaN,2.890928e+05,NaN,NaN,NaN
min,NaN,NaN,NaN,NaN,1.000006e+06,NaN,NaN,NaN
25%,NaN,NaN,NaN,NaN,1.251646e+06,NaN,NaN,NaN
50%,NaN,NaN,NaN,NaN,1.500739e+06,NaN,NaN,NaN
75%,NaN,NaN,NaN,NaN,1.754569e+06,NaN,NaN,NaN


#### How many users have more than one country?

In [238]:
country_combos = (
    df.groupby('user_id')['user_country']
    .agg(lambda x: ' + '.join(sorted(x.unique())))
    .reset_index(name='countries')
)

stage_counts = (
    df.groupby(['user_id', 'event_name'])
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

result = country_combos.merge(stage_counts, on='user_id')

summary = (
    result.groupby('countries')
    .agg(
        unique_users=('user_id', 'nunique'),
        Transfer_Created=('Transfer Created', 'sum'),
        Transfer_Funded=('Transfer Funded', 'sum'),
        Transfer_Transferred=('Transfer Transferred', 'sum'),
    )
    .rename(columns={
        'unique_users': '# Users',
        'Transfer_Created': 'Transfer Created',
        'Transfer_Funded': 'Transfer Funded',
        'Transfer_Transferred': 'Transfer Transferred',
    })
    .sort_values('# Users', ascending=False)
)

summary['Total'] = summary[['Transfer Created', 'Transfer Funded', 'Transfer Transferred']].sum(axis=1)

summary.loc['Total'] = summary.sum()

# Percentage of column total
cols = ['# Users', 'Transfer Created', 'Transfer Funded', 'Transfer Transferred', 'Total']
for col in cols:
    summary[f'{col} %'] = (summary[col] / summary[col].sum() * 100).round(1).astype(str) + '%'

summary.loc['Total'] = summary[cols].sum()

summary

,# Users,Transfer Created,Transfer Funded,Transfer Transferred,Total,# Users %,Transfer Created %,Transfer Funded %,Transfer Transferred %,Total %
countries,,,,,,,,,,
USA,15276,15983,7957,3539,27479,19.0%,18.6%,20.6%,16.0%,18.7%
MEX,15270,15865,5145,3961,24971,19.0%,18.4%,13.3%,17.9%,17.0%
OTHER,9110,9997,5623,3266,18886,11.3%,11.6%,14.6%,14.7%,12.9%
MEX + USA,284,600,269,155,1024,0.4%,0.7%,0.7%,0.7%,0.7%
OTHER + USA,142,319,169,82,570,0.2%,0.4%,0.4%,0.4%,0.4%
MEX + OTHER,139,298,124,75,497,0.2%,0.3%,0.3%,0.3%,0.3%
MEX + OTHER + USA,2,8,3,2,13,0.0%,0.0%,0.0%,0.0%,0.0%
Total,80446,86140,38580,22160,146880,NaN,NaN,NaN,NaN,NaN


In [239]:
monthly = (
    df.assign(month=pd.to_datetime(df['dt']).dt.to_period('M').astype(str))
    .groupby(['month', 'event_name'])
    .size()
    .unstack(fill_value=0)
    .rename(columns={'Transfer Created': 'created', 'Transfer Funded': 'funded', 'Transfer Transferred': 'transferred'})
    .reset_index()
)

monthly['funded_rate']      = (monthly['funded']      / monthly['created']).round(3)
monthly['end_to_end_rate']  = (monthly['transferred'] / monthly['created']).round(3)

monthly[['month', 'created', 'funded', 'transferred', 'funded_rate', 'end_to_end_rate']]

event_name,month,created,funded,transferred,funded_rate,end_to_end_rate
0,2024-01,17470,8660,4666,0.496,0.267
1,2024-02,25600,10630,6404,0.415,0.250
2,2024-03,0,0,10,NaN,inf


In [240]:
df['month'] = pd.to_datetime(df['dt']).dt.to_period('M').astype(str)

#  Base counts per month
base = (
    df.groupby(['month', 'event_name'])
    .size()
    .unstack(fill_value=0)
    .rename(columns={'Transfer Created': 'created', 'Transfer Funded': 'funded', 'Transfer Transferred': 'transferred'})
)

# Funded Rate & End-to-End Conversion
base['funded_rate']     = base['funded']      / base['created']
base['end_to_end_rate'] = base['transferred'] / base['created']

# MoM Growth (Created)
base['mom_growth'] = base['created'].pct_change()

# New User Funded Rate
new_users = (
    df[df['experience'] == 'New']
    .groupby(['month', 'event_name']).size()
    .unstack(fill_value=0)
)
base['new_funded_rate'] = new_users['Transfer Funded'] / new_users['Transfer Created']

#Existing User Funded Rate
existing_users = (
    df[df['experience'] == 'Existing']
    .groupby(['month', 'event_name']).size()
    .unstack(fill_value=0)
)
base['existing_funded_rate'] = existing_users['Transfer Funded'] / existing_users['Transfer Created']

# New vs Existing Gap
base['new_vs_existing_gap'] = base['existing_funded_rate'] - base['new_funded_rate']

# Platform Funded Rate
platform = (
    df.groupby(['month', 'platform', 'event_name']).size()
    .unstack(fill_value=0)
)
for p in df['platform'].unique():
    col = f'funded_rate_{p.lower()}'
    base[col] = platform.loc[(slice(None), p), 'Transfer Funded'].values / platform.loc[(slice(None), p), 'Transfer Created'].values

base.round(3)

C:\Users\g-mac\AppData\Local\Temp\ipykernel_28708\2642101307.py:44: RuntimeWarning:

invalid value encountered in divide



event_name,created,funded,transferred,funded_rate,end_to_end_rate,mom_growth,new_funded_rate,existing_funded_rate,new_vs_existing_gap,funded_rate_ios,funded_rate_web,funded_rate_android
month,,,,,,,,,,,,
2024-01,17470,8660,4666,0.496,0.267,NaN,0.410,0.615,0.205,0.575,0.423,0.493
2024-02,25600,10630,6404,0.415,0.250,0.465,0.307,0.593,0.286,0.489,0.383,0.373
2024-03,0,0,10,NaN,inf,-1.000,NaN,NaN,NaN,NaN,NaN,NaN


### Charts

### Functions

In [241]:
# Safe divides two aligned pandas Series
def safe_divide(numerator: pd.Series, denominator: pd.Series) -> pd.Series:
    result = numerator.div(denominator)
    result = result.replace([np.inf, -np.inf], np.nan)
    return result

# Help to count event in a specific colum
def build_event_count_wide(
    data: pd.DataFrame,
    group_cols: list,
    event_col: str = "event_name"
) -> pd.DataFrame:
    out = (
        data.groupby(group_cols + [event_col])
        .size()
        .unstack(fill_value=0)
        .reset_index()
    )

    # Guarantee expected columns exist even if one event is missing in a slice
    for col in ["Transfer Created", "Transfer Funded", "Transfer Transferred"]:
        if col not in out.columns:
            out[col] = 0

    return out

# fromat number as pecentage 
def to_percentage(value):
    return f"{value:.1%}"

#### 1. Transfers Created | How many  transfers were initiated during per month?

In [242]:
df_transfers_created = (
    df.loc[df["event_name"] == "Transfer Created"]
    .groupby("month", as_index=False)
    .size()
    .rename(columns={"size": "transfers_created"})
)

# Calculate and add a total row
total_row = pd.DataFrame({
    "month": ["Total"],
    "transfers_created": [df_transfers_created["transfers_created"].sum()]
})

df_transfers_created_with_total = pd.concat([df_transfers_created, total_row], ignore_index=True)
df_transfers_created_with_total

,month,transfers_created
0,2024-01,17470
1,2024-02,25600
2,Total,43070


#### 2. Funded Rate | What percentage of created transfers successfully moved to the funded stage?

In [243]:
funded_base = build_event_count_wide(df, ["month"])

df_funded_rate = funded_base[["month", "Transfer Created", "Transfer Funded"]].copy()
df_funded_rate = df_funded_rate.rename(
    columns={
        "Transfer Created": "created_count",
        "Transfer Funded": "funded_count"
    }
)
df_funded_rate["funded_rate"] = safe_divide(
    df_funded_rate["funded_count"],
    df_funded_rate["created_count"]
)

df_funded_rate

event_name,month,created_count,funded_count,funded_rate
0,2024-01,17470,8660,0.495707
1,2024-02,25600,10630,0.415234
2,2024-03,0,0,NaN


#### 3. End-to-End Conversion | What is the overall success rate from the start of a transfer to its completion?

In [244]:
df_end_to_end_conversion = funded_base[
    ["month", "Transfer Created", "Transfer Transferred"]
].copy()

df_end_to_end_conversion = df_end_to_end_conversion.rename(
    columns={
        "Transfer Created": "created_count",
        "Transfer Transferred": "transferred_count"
    }
)
df_end_to_end_conversion["end_to_end_conversion"] = safe_divide(
    df_end_to_end_conversion["transferred_count"],
    df_end_to_end_conversion["created_count"]
)


df_end_to_end_conversion

event_name,month,created_count,transferred_count,end_to_end_conversion
0,2024-01,17470,4666,0.267086
1,2024-02,25600,6404,0.250156
2,2024-03,0,10,NaN


#### 4. MoM Growth (Created) | By what percentage did the volume of created transfers increase or decrease compared to last month?

In [245]:
df_mom_growth_created = df_transfers_created.copy()
df_mom_growth_created = df_mom_growth_created.sort_values("month").reset_index(drop=True)
df_mom_growth_created["mom_growth_created"] = (
    df_mom_growth_created["transfers_created"].pct_change()
)

df_mom_growth_created

,month,transfers_created,mom_growth_created
0,2024-01,17470,NaN
1,2024-02,25600,0.465369


#### 5. New User Funded Rate| How successful are first-time users at getting their transfers funded?

In [246]:
df_new = df.loc[df["experience"] == "New"].copy()
new_base = build_event_count_wide(df_new, ["month"])

df_new_user_funded_rate = new_base[
    ["month", "Transfer Created", "Transfer Funded"]
].copy()

df_new_user_funded_rate = df_new_user_funded_rate.rename(
    columns={
        "Transfer Created": "new_created_count",
        "Transfer Funded": "new_funded_count"
    }
)
df_new_user_funded_rate["new_user_funded_rate"] = safe_divide(
    df_new_user_funded_rate["new_funded_count"],
    df_new_user_funded_rate["new_created_count"]
)

df_new_user_funded_rate

event_name,month,new_created_count,new_funded_count,new_user_funded_rate
0,2024-01,10184,4179,0.410350
1,2024-02,15872,4866,0.306578
2,2024-03,0,0,NaN


#### 6. Existing User Funded Rate |	How successful are returning users at getting their transfers funded?

In [247]:
df_existing = df.loc[df["experience"] == "Existing"].copy()
existing_base = build_event_count_wide(df_existing, ["month"])

df_existing_user_funded_rate = existing_base[
    ["month", "Transfer Created", "Transfer Funded"]
].copy()

df_existing_user_funded_rate = df_existing_user_funded_rate.rename(
    columns={
        "Transfer Created": "existing_created_count",
        "Transfer Funded": "existing_funded_count"
    }
)
df_existing_user_funded_rate["existing_user_funded_rate"] = safe_divide(
    df_existing_user_funded_rate["existing_funded_count"],
    df_existing_user_funded_rate["existing_created_count"]
)
df_existing_user_funded_rate

event_name,month,existing_created_count,existing_funded_count,existing_user_funded_rate
0,2024-01,7286,4481,0.615015
1,2024-02,9728,5764,0.592516
2,2024-03,0,0,NaN


#### 7. Platform Funded Rate | How does the funding success rate differ depending on which platform (app/web) the user is on?

In [248]:
platform_base = build_event_count_wide(df, ["month", "platform"])

# Break down by platform
df_platform_funded_rate = platform_base[
    ["month", "platform", "Transfer Created", "Transfer Funded"]
].copy()

df_platform_funded_rate = df_platform_funded_rate.rename(
    columns={
        "Transfer Created": "platform_created_count",
        "Transfer Funded": "platform_funded_count"
    }
)
df_platform_funded_rate["platform_funded_rate"] = safe_divide(
    df_platform_funded_rate["platform_funded_count"],
    df_platform_funded_rate["platform_created_count"]
)

df_platform_funded_rate 

event_name,month,platform,platform_created_count,platform_funded_count,platform_funded_rate
0,2024-01,Android,6772,3340,0.493207
1,2024-01,Web,5448,2302,0.422540
2,2024-01,iOS,5250,3018,0.574857
3,2024-02,Android,9984,3728,0.373397
4,2024-02,Web,6912,2648,0.383102
5,2024-02,iOS,8704,4254,0.488741
6,2024-03,Android,0,0,NaN
7,2024-03,Web,0,0,NaN
8,2024-03,iOS,0,0,NaN


#### 8. New vs Existing Gap	| What is the performance difference in funding rates between existing users and new users?

In [249]:
# Keep only the columns needed for the comparison
df_new_vs_existing_gap = (
    df_existing_user_funded_rate[["month", "existing_user_funded_rate"]]
    .merge(
        df_new_user_funded_rate[["month", "new_user_funded_rate"]],
        on="month",
        how="outer"
    )
    .sort_values("month")
    .reset_index(drop=True)
)

# Force numeric types in case the rate columns came as strings/object
df_new_vs_existing_gap["existing_user_funded_rate"] = pd.to_numeric(
    df_new_vs_existing_gap["existing_user_funded_rate"],
    errors="coerce"
)

df_new_vs_existing_gap["new_user_funded_rate"] = pd.to_numeric(
    df_new_vs_existing_gap["new_user_funded_rate"],
    errors="coerce"
)

# Gap = Existing funded rate - New funded rate
df_new_vs_existing_gap["new_vs_existing_gap"] = (
    df_new_vs_existing_gap["existing_user_funded_rate"]
    - df_new_vs_existing_gap["new_user_funded_rate"]
)

df_new_vs_existing_gap

event_name,month,existing_user_funded_rate,new_user_funded_rate,new_vs_existing_gap
0,2024-01,0.615015,0.410350,0.204666
1,2024-02,0.592516,0.306578,0.285939
2,2024-03,NaN,NaN,NaN


## Charts

In [250]:
df_w = df.copy()
df_w['dt'] = pd.to_datetime(df_w['dt'])
df_w['week_start'] = df_w['dt'] - pd.to_timedelta(df_w['dt'].dt.dayofweek, unit='D')

weekly = (
    build_event_count_wide(df_w, ['week_start'])
    .rename(columns={'Transfer Created': 'created', 'Transfer Funded': 'funded'})
    .sort_values('week_start')
    .reset_index(drop=True)
)
weekly['funded_rate'] = safe_divide(weekly['funded'], weekly['created'])
weekly9 = weekly[weekly['week_start'] >= '2024-01-01'].head(9).reset_index(drop=True)
weekly9['label'] = [
    'W1 Jan', 'W2 Jan', 'W3 Jan', 'W4 Jan', 'W5 Jan/Feb',
    'W6 Feb', 'W7 Feb', 'W8 Feb', 'W9 Feb/Mar',
]

fig = go.Figure(go.Scatter(
    x=weekly9['label'],
    y=weekly9['funded_rate'],
    mode='lines+markers+text',
    text=[f'{r:.1%}' for r in weekly9['funded_rate']],
    textposition='top center',
    fill='tozeroy',
    fillcolor='rgba(99, 110, 250, 0.15)',
    line=dict(color='rgb(99, 110, 250)'),
    marker=dict(size=9),
))

fig.update_layout(
    plot_bgcolor='rgba(0,0,0,0)',  # Transparent background inside the grid
    paper_bgcolor='rgba(0,0,0,0)', # Transparent background for the whole image
    title='Weekly Funded Rate',
    xaxis_title='Week',
    yaxis=dict(
        title='Funded Rate',
        tickformat='.0%',
        range=[0, 1],
    ),
    xaxis=dict(
        range=[-0.5, 8.5], 
        tickangle=-45,          
        automargin=True         
    ),
    margin=dict(l=80, r=80, t=100, b=120, pad=10),
    autosize=False, # Forces Plotly to respect the pixel dimensions if provided
    width=900,      # Optional: defining a width helps visualize the margin
    height=500
)
fig.show()


In [ ]:
monthly_counts = (
    df[df['month'].isin(['2024-01', '2024-02'])]
    .groupby(['month', 'event_name'])
    .size()
    .reset_index(name='Count')
)

event_map = {
    'Transfer Created': 'Transfer Created',
    'Transfer Funded': 'Transfer Funded',
    'Transfer Transferred': 'Transfer Transferred'
}
monthly_counts = monthly_counts[monthly_counts['event_name'].isin(event_map.keys())]
monthly_counts['Event'] = monthly_counts['event_name'].map(event_map)


monthly_counts['Month'] = monthly_counts['month'].map({'2024-01': 'January', '2024-02': 'February'})


custom_colors = {'January': '#4B6B2F', 'February': '#345E61'}

fig = px.bar(
    monthly_counts,
    x='Event', 
    y='Count', 
    color='Month',
    barmode='group',
    text_auto=',.0f', 
    color_discrete_map=custom_colors,
    template='plotly_dark'
)

# Styling 
fig.update_layout(
    title=dict(
        text='Monthly Conversion Pipeline',
        x=0.05, y=0.95,
        font=dict(family="Arial, sans-serif", size=24, color='#88A070')
    ),
    paper_bgcolor='rgba(0,0,0,0)', 
    plot_bgcolor='rgba(0,0,0,0)',  
    bargap=0.3,       
    bargroupgap=0.1,  
    
    legend=dict(
        orientation="h", yanchor="bottom", y=1.05,
        xanchor="center", x=0.5, title_text=""
    ),
    
    xaxis=dict(
        showgrid=False, title="",
        tickfont=dict(color='#88A070')
    ),
    yaxis=dict(
        gridcolor='#1A2610', title="",
        tickfont=dict(color='#88A070'),
        range=[0, monthly_counts['Count'].max() * 1.2] 
    ),
    margin=dict(t=100, b=50, l=50, r=50)
)

fig.update_traces(
    marker_cornerradius=10, 
    marker_line_width=1,
    marker_line_color='#88A070',
    textposition='outside',
    textfont=dict(color='#88A070')
)

fig.show()